# DS 208 · Week 6 — pandas I
## Loading, selecting and filtering

Every code block from the week 6 lecture, in order, so you can run what you watched
instead of retyping it. Parts A–C use a five-city table typed into the first cell; Part D
loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). After
that, change things: the point of a notebook is that you can break it and find out why.

**Version differences.** The deck follows pandas 3, which made copy-on-write permanent.
Check yours with `pd.__version__`. On pandas 2, text columns show as `object` instead of
`str`, memory figures differ, and the two copy slides in Part C raise a
`SettingWithCopyWarning`.

A section at the end, **Beyond the lecture**, takes the same tools further on the flood
data.


In [ ]:
import pandas as pd

# Show every result in a cell, not only the last one, so each cell prints what its
# slide shows.
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

---
## Part 1 — The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter, so you can jump to whichever
one you want to replay.


### Slide 7 — pandas also runs on these slides


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 9 — read_csv parses the file and infers types


Expected output:

```
city    region      pop
0      Cebu   Visayas   964000
1     Davao  Mindanao  1777000
2    Iloilo   Visayas   457000
3    Baguio     Luzon   366000
4  Tacloban   Visayas   251000
(5, 3)
```


In [ ]:
import pandas as pd
import io

csv_text = """city,region,pop
Cebu,Visayas,964000
Davao,Mindanao,1777000
Iloilo,Visayas,457000
Baguio,Luzon,366000
Tacloban,Visayas,251000"""

df = pd.read_csv(io.StringIO(csv_text))
print(df.head())
print(df.shape)

### Slide 10 — Four checks for every new dataset


Expected output:

```
['city', 'region', 'pop']
city        str
region      str
pop       int64
dtype: object
(5, 3)
pop
count        5.0
mean    763000.0
std     628885.0
min     251000.0
25%     366000.0
50%     457000.0
75%     964000.0
max    1777000.0
```


In [ ]:
print(df.columns.tolist())
print(df.dtypes)
print(df.shape)
print(df.describe().round())

### Slide 13 — Single and double brackets


Expected output:

```
0     964000
1    1777000
2     457000
3     366000
4     251000
Name: pop, dtype: int64
city      pop
0      Cebu   964000
1     Davao  1777000
2    Iloilo   457000
3    Baguio   366000
4  Tacloban   251000
```


In [ ]:
print(df["pop"])
print(df[["city", "pop"]])

### Slide 15 — Labels compared with positions


**Exercise.**

1. Predict how many rows each of the last two lines prints.
2. Run the cell.
3. Change both slices to 1:3. Which cities does each return now?


In [ ]:
print(df.loc[0, "region"])      # row label 0, column "region"
print(df.iloc[0, 1])            # position 0, column position 1

print(df.loc[0:2, ["city", "pop"]])
print(df.iloc[0:2, [0, 2]])

### Slide 16 — Index labels are preserved after filtering


Expected output:

```
[2, 3, 4]
Iloilo
KeyError: 0
[0, 1, 2]
```


In [ ]:
small = df[df["pop"] < 500_000]
print(small.index.tolist())

print(small.iloc[0]["city"])    # position 0: works
try:
    small.loc[0]                # label 0: no such row now
except KeyError as e:
    print("KeyError:", e)

small = small.reset_index(drop=True)
print(small.index.tolist())

### Slide 17 — A single row or all matching rows


Expected output:

```
city         Cebu
region    Visayas
pop        964000
Name: 0, dtype: object
city      Tacloban
region     Visayas
pop         251000
Name: 4, dtype: object
city   region     pop
0  Cebu  Visayas  964000
```


In [ ]:
print(df.loc[0])                # the row labelled 0
print(df.iloc[-1])              # the row in the last position

print(df.loc[df["city"] == "Cebu"])

### Slide 21 — A condition selects rows


Expected output:

```
0    False
1     True
2    False
3    False
4    False
Name: pop, dtype: bool
city    region      pop
1  Davao  Mindanao  1777000
```


In [ ]:
print(df["pop"] > 1_000_000)

print(df[df["pop"] > 1_000_000])

### Slide 22 — Combine with & and |


**Exercise.**

1. Predict which cities each of the two printed frames contains.
2. Run the code.
3. Change 500_000 to 400_000. Which of the two results changes?


In [ ]:
big = df["pop"] > 500_000
vis = df["region"] == "Visayas"

print(df[big & vis])
print(df[big | vis])

### Slide 26 — Editing a filtered frame does not change the original


Expected output:

```
sub: [99, 99]
t  : [1, 2, 3, 4]
```


In [ ]:
t = pd.DataFrame({"g": ["x", "x", "y", "y"],
                  "a": [1, 2, 3, 4]})

sub = t[t.g == "x"]
sub["a"] = 99

print("sub:", sub.a.tolist())
print("t  :", t.a.tolist())

### Slide 27 — Assigning through .loc on the target frame


Expected output:

```
t: [77, 77, 3, 4]
```


In [ ]:
# t: the 4-row frame from the previous slide
sub = t[t.g == "x"]
sub["a"] = 99                   # changes sub only

t.loc[t.g == "x", "a"] = 77     # changes t itself

print("t:", t.a.tolist())

### Slide 30 — Counting categories and summarising numbers


**Exercise.**

1. Predict the most common region and its count.
2. Run the code.
3. Change .mean() to .median(). Why is the result smaller?


In [ ]:
print(df["region"].value_counts())

print(df["pop"].mean())

### Slide 31 — Finding the largest Visayan city


**Exercise.**

1. Predict which city the last line prints.
2. Run the code.
3. Change .tail(1) to .head(1). Which city now, and why?


In [ ]:
vis = df[df["region"] == "Visayas"]

print(vis["pop"].mean())
print(vis.sort_values("pop").tail(1))

### Slide 33 — From the course API to a DataFrame


Expected output:

```
(34079, 15)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
resp = requests.get(url, timeout=60)
df   = pd.read_csv(io.StringIO(resp.text))

df.shape

### Slide 34 — .info() reports types and memory in one call


Expected output:

```
RangeIndex: 34079 entries, 0 to 34078
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype
 0   contract_id  34079 non-null  str
 4   budget       34079 non-null  float64
 8   start_date   33106 non-null  str  <-
 10  year         34079 non-null  int64
memory usage: 13.0 MB
```


In [ ]:
df.info()

# excerpt: 4 of the 15 column rows

### Slide 35 — Converting three columns to category saves 2.3 MB


Expected output:

```
13.644672     # MB, as loaded
11.314324     # 17% smaller
```


In [ ]:
df.memory_usage(deep=True).sum() / 1e6


for c in ["status", "region", "category"]:
    df[c] = df[c].astype("category")

df.memory_usage(deep=True).sum() / 1e6


# and groupby gets faster too

### Slide 36 — Converting types at load time


Expected output:

```
(0, 973)
```


In [ ]:
df = pd.read_csv(io.StringIO(resp.text))
df["budget"] = pd.to_numeric(df.budget, errors="coerce")
df["start_date"] = pd.to_datetime(
    df.start_date, errors="coerce")

# how many refused?
df.budget.isna().sum(), df.start_date.isna().sum()


# 973 = the rows that were already blank

### Slide 37 — String methods applied down a column


Expected output:

```
22,022        # 65% of all projects
2,782
139.43490125883974  # chars
```


In [ ]:
df.description.str.contains("FLOOD",
                            case=False, na=False).sum()


df.description.str.contains("REVETMENT", na=False).sum()


df.description.str.len().mean()


# also: .lower() .strip() .split()
#       .replace() .startswith()

### Slide 38 — Date components with .dt


Expected output:

```
2015-05-17   2025-09-23
1.0     1054
2.0     9416   <-
3.0     7122   <-
4.0     3972
...
12.0    1041
```


In [ ]:
df.start_date.min(), df.start_date.max()


df.start_date.dt.month.value_counts().sort_index()

### Slide 40 — value_counts with normalize=True


Expected output:

```
Completed          27534
On-Going            5442
For Procurement      876
Completed          0.8079
On-Going           0.1597
For Procurement    0.0257
```


In [ ]:
df.status.value_counts()


df.status.value_counts(normalize=True)


# 80.8% — the honest headline figure

### Slide 41 — nlargest for the top rows


Expected output:

```
budget          region
 1,447,499,996  Central Office
 1,096,255,230  Central Office
 1,042,782,802  Central Office
```


In [ ]:
df.nlargest(3, "budget")[["budget", "region"]]


# ascending, for the other end
df.nsmallest(3, "budget")

### Slide 42 — The same filter written as a string


In [ ]:
# bracket form
df[(df.region == "Region III") &
   (df.budget > 1e8) &
   (df.status == "Completed")]

# query form — same result
df.query("region == 'Region III' "
         "and budget > 1e8 "
         "and status == 'Completed'")

# with a variable
floor = 1e8
df.query("budget > @floor")

### Slide 43 — The Series truth-value error


Expected output:

```
ValueError: The truth value of a
Series is ambiguous. Use a.empty,
a.bool(), a.item(), a.any() or a.all()
```


This cell contains a line that **fails on purpose**. It is wrapped in `try`/`except` (week 4) so the notebook keeps running; the error message is printed instead.


In [ ]:
# wrong
try:
    df[df.budget > 1e8 and df.status == "Completed"]
except ValueError as e:
    print("ValueError:", e)


# right — note the parentheses
df[(df.budget > 1e8) &
   (df.status == "Completed")]

# also wrong: missing parens
try:
    df[df.budget > 1e8 & df.status == ...]
except TypeError as e:
    print("TypeError:", e)
# & runs first, on the numbers

### Slide 44 — describe() output depends on dtype


Expected output:

```
count          34079
unique            18
top       Region III
freq            5412
```


In [ ]:
df.region.describe()


# 5412/34079 = 16% in one region

df.describe()        # numeric only, by default
df.describe(include="all")   # everything

### Slide 45 — fillna takes a value, a method, or a rule


In [ ]:
# a constant
df.contractor.fillna("Not yet awarded")

# a computed value
df.budget.fillna(df.budget.median())

# per group — usually more honest
df.budget.fillna(
    df.groupby("region").budget.transform("median"))

# count what you changed
before = df.budget.isna().sum()

### Slide 46 — apply is more than 100× slower for the same arithmetic


**Exercise.**

1. Predict how many times slower apply is.
2. Run the code. True means both give the same answers.
3. Change 34_079 to 1_000. Does the gap stay the same?


Timings differ from machine to machine; compare the ratio, not the exact numbers. The lecture laptop measured 2.2 ms against 0.013 ms.


In [ ]:
import numpy as np
import timeit

def time_ms(f):
    """Average milliseconds per call of f()."""
    calls, seconds = timeit.Timer(f).autorange()
    return seconds / calls * 1000

# time_ms(f) is week 5's timeit helper
rng = np.random.default_rng(6)
budget = pd.Series(rng.uniform(1e6, 1e9, 34_079))   # made-up

def with_apply():
    return budget.apply(lambda x: x / 1e6)

def vectorised():
    return budget / 1e6

print(with_apply().equals(vectorised()))
print(f"apply     : {time_ms(with_apply):.2f} ms")
print(f"vectorised: {time_ms(vectorised):.3f} ms")

---
# Part 2 — Beyond the lecture

Everything below is new, and uses only this week's tools: filters, `&`, `.loc`/`.iloc`,
and `value_counts`. Three questions about the flood data.

The cell below reloads it, so this part runs even if you skipped Part 1.


In [ ]:
import io, requests

resp = requests.get("https://portal.latarak.com/api/flood/export.csv", timeout=60)
df = pd.read_csv(io.StringIO(resp.text))
df.shape

## 1. The index trap, on real data

Part B showed that a filter keeps the original row labels. On 34,079 rows the labels are
large numbers, and `.loc[0]` on a filtered frame fails for the same reason it did on the
cities.


In [ ]:
big = df[df.budget > 1e9]
big.index

big.iloc[0]["contract_id"]    # position 0: works
try:
    big.loc[0]                # label 0: not in this frame
except KeyError as e:
    print("KeyError:", e)

The three billion-peso projects keep labels 18777, 28049 and 33185, their rows in the
full table. That is useful (each row can be traced back) and a trap (`.loc[0]` asks for a
label that is not there).


## 2. Do `status` and `progress` agree?

Two columns describe the same thing: whether a project is finished. Combining conditions
with `&` checks whether they agree.


In [ ]:
done_but_not_100 = df[(df.status == "Completed") & (df.progress < 100)]
going_but_100    = df[(df.status == "On-Going")  & (df.progress == 100)]

len(done_but_not_100), len(going_but_100), (df.status == "On-Going").sum()

Only 10 "Completed" projects report less than 100% progress, but **972 of the 5,442
"On-Going" projects report 100%**. Either the status lags behind the progress figure or
the progress figure runs ahead of the work. The data cannot say which.

The finding is the disagreement itself. Any count of "unfinished projects" depends on
which column you trust, so state which one you used.


## 3. Where are the projects not yet awarded?

876 projects are **For Procurement**: no contractor yet. Counting them per region is one
`value_counts`. Dividing by each region's total gives a share, and pandas matches the two
Series **by label** (region name), not by position.


In [ ]:
fp = df[df.status == "For Procurement"]

counts = fp.region.value_counts()
share  = counts / df.region.value_counts()     # aligned by region name

counts.head(3)
share.sort_values(ascending=False).head(3).round(3)

Region IV-B has the most projects awaiting award (132), and also the highest share:
about 9% of its projects, against 2.6% nationally. The largest count and the largest share
need not belong to the same region, so check both before writing "most".

---

## What to take from this

1. **A filter keeps the original labels.** `.iloc` counts positions; `.loc` looks up labels
   that may no longer exist.
2. **Two columns can disagree.** `&` filters are the quickest way to find out, and the
   result decides which column a later count should use.
3. **pandas aligns by label.** Dividing two Series matched regions by name, which is why
   the share came out right even though the two counts were in different orders.

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at*
`/api/flood` *— see* `/api/flood` *itself for the full endpoint list.*
